# Point-pattern second-order statistics lab
Explore nearest-neighbor distances, homogeneous and inhomogeneous rectangular translation-corrected Ripley's K, and pair correlation. All coordinates are planar and use the same linear unit. The inhomogeneous estimator requires externally supplied intensity estimates at the observed points.


In [ ]:
import sys
sys.path.insert(0, "implementations/python/point-process")
from point_patterns import nearest_neighbor_distances, nearest_neighbor_cdf, ripley_k_translation, ripley_k_inhomogeneous_translation, pair_correlation_from_k, quadrat_count_test
import numpy as np
from spatial_scan import poisson_spatial_scan, monte_carlo_poisson_scan
import math

points = [(2,2), (3,2), (8,7), (8,8), (5,5), (6,5)]
radii = [0.5, 1, 2, 3, 4]
print('Nearest-neighbor distances:', nearest_neighbor_distances(points))
print('Nearest-neighbor CDF:', nearest_neighbor_cdf(points, radii))
k = ripley_k_translation(points, (0,0,10,10), radii)
print('K(r):', k)
print('Poisson reference:', [math.pi*r*r for r in radii])
print('Approximate g(r):', pair_correlation_from_k(radii, k))


## Quadrat counts: a grid-scale CSR diagnostic

The conditional Monte Carlo test compares equal-area quadrat counts with a uniform multinomial null, given the observed total point count. The grid is a modelling choice: different grid sizes or alignments can produce different results.

In [ ]:
quadrat = quadrat_count_test(points, (0, 0, 10, 10), nx=3, ny=3, n_simulations=499, seed=2026)
print('counts by row:', quadrat['counts'])
print('expected per quadrat:', quadrat['expected_per_quadrat'])
print('Pearson statistic:', quadrat['pearson_statistic'])
print('conditional Monte Carlo p-value:', quadrat['pvalue'])


## Inhomogeneous K: intensity reweighting

Here the intensity values are supplied at the observed points. They are illustrative inputs, not estimates fitted from this tiny pattern. In real analysis, fit a defensible trend or use a leave-one-out estimator and assess sensitivity to its bandwidth.

In [ ]:
intensity_at_points = [0.012, 0.018, 0.010, 0.014, 0.020, 0.016]
k_inhom = ripley_k_inhomogeneous_translation(points, (0,0,10,10), radii, intensity_at_points)
print('Inhomogeneous K(r):', k_inhom)
print('Inhomogeneous Poisson reference:', [math.pi*r*r for r in radii])
print('Approximate inhomogeneous g(r):', pair_correlation_from_k(radii, k_inhom))


## Interpretation and cautions

1. Why must the intensity be evaluated at each observed point?
2. What happens if the fitted intensity surface absorbs genuine clustering?
3. Why does the translation correction depend on pair displacement?
4. Why should the observed K curve be compared with simulations under a fitted null model?

The implementation assumes a rectangular window and second-order intensity-reweighted stationarity. It does not estimate intensity, and a visually close match to the Poisson reference is not a formal test.

## Circular Poisson spatial scan

This baseline searches centroid-centred circles for elevated case rates while using population as the exposure offset. The example is synthetic and uses projected planar coordinates.

In [ ]:
scan_coordinates = [(0,0), (1,0), (10,0), (11,0), (20,0)]
scan_cases = [9, 8, 1, 1, 1]
scan_population = [100, 100, 100, 100, 100]
scan = poisson_spatial_scan(scan_coordinates, scan_cases, scan_population, max_population_fraction=0.5)
print('most likely zone:', scan['member_indices'])
print('log likelihood ratio:', scan['log_likelihood_ratio'])
print('relative risk:', scan['relative_risk'])
mc = monte_carlo_poisson_scan(scan_coordinates, scan_cases, scan_population, n_simulations=99, rng=np.random.default_rng(7))
print('conditional multinomial Monte Carlo p-value (99 simulations):', mc['p_value'])


The scan conditions on total cases and redistributes them across areas in proportion to population under the null. It searches circles centred on the supplied centroids and is limited by the maximum population fraction. It is not covariate-adjusted, time-dependent, elliptical or a full SaTScan implementation. The 99-simulation example is illustrative, not a high-resolution inferential analysis.

## Questions
1. How does the translation correction change pairs near the window boundary?
2. Why is K(r) cumulative while g(r) is local in scale?
3. What changes when the point intensity varies across the study window?
4. Why does a population offset matter for the spatial scan statistic?
5. Why is a Monte Carlo p-value needed after maximising over many candidate circles?


## Cross-type summaries for marked patterns
Compare events of type A around type B events. Cross-G is directional; cross-K uses a rectangular translation correction.

In [ ]:
from point_patterns import cross_nearest_neighbor_cdf, cross_ripley_k_translation, cross_pair_correlation_from_k, cross_j_function
marked_points = [(2,2), (7,7), (3,2), (7,8)]
marks = ['A', 'A', 'B', 'B']
cross_radii = [0.5, 1, 2, 3]
g_ab = cross_nearest_neighbor_cdf(marked_points, marks, 'A', 'B', cross_radii)
k_ab = cross_ripley_k_translation(marked_points, marks, (0,0,10,10), 'A', 'B', cross_radii)
print('G_A_to_B:', g_ab)
print('K_A_to_B:', k_ab)
print('approximate cross g:', cross_pair_correlation_from_k(cross_radii, k_ab))
print('cross J:', cross_j_function(marked_points, marks, (0,0,10,10), 'A', 'B', cross_radii, n_samples=999, seed=2026))


Cross-type functions describe dependence between component patterns. Under stationary independence, K_ij(r)=πr², pair correlation is 1 and J_ij(r)=1. The included cross-G and F/J estimators are simple finite-window teaching baselines; do not treat them as fully edge-corrected inference.

## Point-process model simulation: Poisson and Thomas
The homogeneous Poisson process is a CSR baseline. The Thomas process generates latent Poisson parents and Gaussian-displaced Poisson offspring.

In [ ]:
from point_patterns import simulate_homogeneous_poisson, simulate_thomas_process
window = (0, 0, 10, 10)
csr = simulate_homogeneous_poisson(window, intensity=0.08, seed=2026)
clustered = simulate_thomas_process(window, parent_intensity=0.015, mean_offspring=5, sigma=0.45, seed=2026)
print('homogeneous Poisson event count:', len(csr))
print('Thomas offspring count:', len(clustered))
print('Thomas implied stationary intensity:', 0.015 * 5)


The Thomas simulator generates parents in a buffer-expanded window because parents outside the observation window can have offspring inside it. Its finite Gaussian buffer truncates a small tail. These are simulators, not fitted models or goodness-of-fit tests.

## Matérn disc-cluster simulation
The Matérn model uses Poisson parent centres and Poisson offspring uniformly distributed within a fixed-radius disc. The stationary intensity is κμ. Unlike the Gaussian Thomas model, the offspring displacement has compact support, so a parent-window buffer equal to the disc radius is sufficient.


In [ ]:
from point_patterns import simulate_matern_cluster_process
matern = simulate_matern_cluster_process(window, parent_intensity=0.015, mean_offspring=5, radius=0.7, seed=2026)
print('Matérn offspring count:', len(matern))
print('Matérn implied stationary intensity:', 0.015 * 5)
print('All offspring inside window:', all(0 <= x <= 10 and 0 <= y <= 10 for x, y in matern))


The Matérn and Thomas examples share the same parent intensity and mean offspring count, but use different displacement kernels. Compare their cluster radii and realised patterns across several seeds; one realisation is not a model-comparison test. The simulator returns offspring only and does not expose latent parent identities.


## Log-Gaussian Cox process simulation
The LGCP uses an exponential-covariance Gaussian log-intensity field sampled at regular cell centres. Conditional Poisson counts are generated per cell. The result is a piecewise-constant grid approximation, not an exact continuous-field realisation.


In [ ]:
from point_patterns import simulate_log_gaussian_cox_process
lgcp = simulate_log_gaussian_cox_process(window, mean_intensity=0.8, variance=0.6, correlation_range=2.0, nx=6, ny=6, seed=2026)
print('LGCP event count:', len(lgcp))
print('All events inside window:', all(0 <= x <= 10 and 0 <= y <= 10 for x, y in lgcp))


The variance parameter controls latent log-intensity variation; the correlation range controls its spatial smoothness. The -variance/2 correction preserves the requested marginal mean intensity. Dense Cholesky factorisation limits this teaching implementation to at most 225 cells. Do not infer model fit or statistical significance from one simulated pattern.


## Strauss repulsive point process
The Strauss model penalises close pairs. The parameter γ=1 is the homogeneous Poisson limit; γ=0 imposes a hard-core distance. The following finite-window birth-death Metropolis–Hastings run is approximate and should be lengthened and diagnosed for serious use.


In [ ]:
from point_patterns import simulate_strauss_process
strauss = simulate_strauss_process(window, beta=0.06, gamma=0.25, interaction_radius=0.6, burn_in=3000, n_steps=5000, seed=2026)
print('Strauss event count:', len(strauss))
print('All events inside window:', all(0 <= x <= 10 and 0 <= y <= 10 for x, y in strauss))


In [ ]:
# Hard-core limit: no returned pair should be closer than the interaction radius.
hard_core = simulate_strauss_process(window, beta=0.08, gamma=0, interaction_radius=0.6, burn_in=3000, n_steps=5000, seed=2027)
minimum_pair_distance = min(( ((x-a)**2 + (y-b)**2)**0.5 for i, (x, y) in enumerate(hard_core) for a, b in hard_core[i+1:] ), default=float('inf'))
print('Hard-core minimum pair distance:', minimum_pair_distance)


A hard-core constraint check verifies the support of the sampled configurations, not MCMC convergence. The activity β, interaction γ, radius, initialisation and chain length all affect mixing and realised event counts.


## Besag–Newell fixed-k cluster detection
For each area centroid, accumulate nearest areas until at least k cases are reached. The exact local tail conditions on total cases and population. The global Monte Carlo test repeats the entire scan under population-proportional case allocation.


In [ ]:
from spatial_scan import besag_newell_scan, monte_carlo_besag_newell
centroids = np.array([[0., 0.], [1., 0.], [-1., 0.], [10., 0.], [11., 0.]])
cases = np.array([8, 5, 5, 1, 1])
population = np.array([100, 100, 100, 100, 100])
bn = besag_newell_scan(centroids, cases, population, k=10, alpha=0.05)
[(r['center_index'], r['n_areas'], r['cases_inside'], round(r['p_value'], 4)) for r in bn['local_results']]


In [ ]:
bn_global = monte_carlo_besag_newell(centroids, cases, population, k=10,
                                     alpha=0.05, n_simulations=199, rng=42)
bn_global['observed']['n_significant'], bn_global['p_value']


## Tango global clustering test
Tango's quadratic form weights population-adjusted case-proportion residuals by an exponential centroid-distance matrix. The Monte Carlo null fixes total cases and reallocates them in proportion to population.


In [ ]:
from spatial_scan import tango_statistic, monte_carlo_tango_test
tango = tango_statistic(centroids, cases, population, phi=2.0)
tango['statistic'], tango['closeness_matrix']


In [ ]:
tango_test = monte_carlo_tango_test(centroids, cases, population, phi=2.0,
                                    n_simulations=199, rng=42)
tango_test['observed']['statistic'], tango_test['p_value']


## Elliptical Poisson spatial scan
The scan searches centroid-centred ellipses over a finite grid of axis ratios and orientations. The global Monte Carlo test repeats the entire search under the conditional constant-risk null.


In [ ]:
from spatial_scan import elliptical_poisson_scan, monte_carlo_elliptical_poisson_scan
elliptical = elliptical_poisson_scan(
    centroids, cases, population, shape_ratios=(1., 2., 3.),
    angles_degrees=(0., 45., 90., 135.), max_population_fraction=0.75
)
elliptical


In [ ]:
elliptical_test = monte_carlo_elliptical_poisson_scan(
    centroids, cases, population, n_simulations=199,
    shape_ratios=(1., 2., 3.), angles_degrees=(0., 45., 90., 135.),
    max_population_fraction=0.75, rng=42
)
elliptical_test['observed']['log_likelihood_ratio'], elliptical_test['p_value']


## Flexible connected-zone scan
This bounded FleXScan-style baseline enumerates connected subsets inside a root area's K-nearest-centroid neighbourhood. Connectivity comes from an explicit symmetric adjacency graph; Monte Carlo inference repeats the complete search.


In [ ]:
from spatial_scan import flexible_poisson_scan, monte_carlo_flexible_poisson_scan
# Example adjacency graph: a simple chain of five areas.
chain_adjacency = np.zeros((5, 5), dtype=bool)
for i in range(4):
    chain_adjacency[i, i + 1] = chain_adjacency[i + 1, i] = True
flexible = flexible_poisson_scan(
    centroids[:5], cases[:5], population[:5], chain_adjacency,
    max_cluster_size=3, max_population_fraction=0.8
)
flexible


In [ ]:
flexible_test = monte_carlo_flexible_poisson_scan(
    centroids[:5], cases[:5], population[:5], chain_adjacency,
    max_cluster_size=3, max_population_fraction=0.8,
    n_simulations=199, rng=42
)
flexible_test['observed']['log_likelihood_ratio'], flexible_test['p_value']
